# Iceberg 1.12: semi-structured data with the `VARIANT` type (format v3)

Iceberg's **v3 table format** adds a `variant` column type for JSON-like data whose shape varies from row to row. Iceberg 1.12 improves it for Spark 4.0/4.1
(vectorized Parquet reads for unshredded variants, many variant-shredding fixes). Spark 4.1 understands `VARIANT` natively, and so does Trino 483.

You will
1. create a **format-version 3** table with a `VARIANT` column in the lab's Iceberg REST catalog,
2. store events with different JSON shapes and query them with Spark's path syntax and variant functions,
3. read the same table from Trino,
4. turn on **variant shredding** and look inside the Parquet files, including a known 1.12.0 limitation and its workaround.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from lab_utils import get_spark, trino, parquet_footer, try_sql, CATALOG

spark = get_spark("iceberg-variant")
print("Spark", spark.version)
NS = f"{CATALOG}.iceberg12"
_ = spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")

## 1. A v3 table with a `VARIANT` column
`format-version = 3` is what enables the new types. Everything else is ordinary Iceberg.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {NS}.events_v3")
spark.sql(f"""CREATE TABLE {NS}.events_v3 (id BIGINT, kind STRING, payload VARIANT) USING iceberg
              TBLPROPERTIES ('format-version' = '3')""")
spark.sql(f"DESCRIBE {NS}.events_v3").show(truncate=False)
spark.sql(f"SHOW TBLPROPERTIES {NS}.events_v3 ('format-version')").show()

## 2. Three event types, three different shapes
No schema migration is needed when a new event type shows up with different fields: that is the point of `VARIANT`.
`parse_json` turns JSON text into a variant value.

In [ ]:
spark.sql(f"""INSERT INTO {NS}.events_v3 VALUES
 (1, 'click',    parse_json('{{"user":{{"id":7,"plan":"pro"}},"target":"btn-buy","ms":120}}')),
 (2, 'purchase', parse_json('{{"user":{{"id":8,"plan":"free"}},"items":[{{"sku":"A1","qty":2,"price":9.5}},{{"sku":"B2","qty":1,"price":20.0}}]}}')),
 (3, 'view',     parse_json('{{"user":{{"id":9}},"url":"/home","tags":["new","mobile"]}}'))""")
spark.sql(f"SELECT id, kind, to_json(payload) AS json FROM {NS}.events_v3 ORDER BY id").show(truncate=False)

### Query inside the variant
* `payload:user.plan::string` is the **path syntax**: `:` navigates, `::type` casts. Missing paths give `NULL`, not an error.
* `variant_get(payload, '$.items[0].sku', 'string')` is the function form (JSONPath).
* `schema_of_variant_agg(...)` infers the union of all shapes in the column: handy to see what has actually been stored.
* `variant_explode` turns an array inside a variant into rows.

In [ ]:
spark.sql(f"SELECT id, payload:user.plan::string AS plan, payload:ms::int AS ms FROM {NS}.events_v3 ORDER BY id").show()
spark.sql(f"SELECT id, variant_get(payload, '$.items[0].sku', 'string') AS first_sku FROM {NS}.events_v3 WHERE kind = 'purchase'").show()
spark.sql(f"SELECT schema_of_variant_agg(payload) AS all_shapes FROM {NS}.events_v3").show(truncate=False)
spark.sql(f"""SELECT id, v:sku::string AS sku, v:qty::int AS qty, v:price::double AS price
              FROM {NS}.events_v3, LATERAL variant_explode(payload:items) AS t(pos, key, v) WHERE kind = 'purchase'""").show()

## 3. Trino reads the same table
Trino 483 exposes the column as type `variant` and returns it as JSON text; cast to `JSON` to use Trino's JSON functions.

In [ ]:
print(trino("DESCRIBE iceberg.iceberg12.events_v3").to_string(index=False))
print()
print(trino("""SELECT id, kind, json_extract_scalar(CAST(payload AS JSON), '$.user.plan') AS plan
                FROM iceberg.iceberg12.events_v3 ORDER BY id""").to_string(index=False))

## 4. Variant shredding
By default a variant is stored as two binary columns (`metadata`, `value`) that must be decoded to read one field.
With **shredding** (`write.parquet.shred-variants = true`) Iceberg infers the common fields from the first rows of each file
(`write.parquet.variant-inference-buffer-size`, default 100) and also writes them as **typed Parquet columns**, which are smaller and can be read/pruned directly.

Below, the same 50,000 events are written twice, with and without shredding, and the Parquet footers are compared.

In [ ]:
for name, shred in (("events_plain", "false"), ("events_shredded", "true")):
    spark.sql(f"DROP TABLE IF EXISTS {NS}.{name}")
    spark.sql(f"""CREATE TABLE {NS}.{name} (id BIGINT, payload VARIANT) USING iceberg
                  TBLPROPERTIES ('format-version' = '3', 'write.parquet.shred-variants' = '{shred}')""")
    spark.sql(f"""INSERT INTO {NS}.{name}
        SELECT id, parse_json(concat('{{"user":{{"id":', id % 100, ',"plan":"', CASE WHEN id % 3 = 0 THEN 'pro' ELSE 'free' END,
                                      '"}},"ms":', id % 997, ',"ok":true}}'))
        FROM range(0, 50000)""")
    size = spark.sql(f"SELECT sum(file_size_in_bytes) FROM {NS}.{name}.files").collect()[0][0]
    path = spark.sql(f"SELECT file_path FROM {NS}.{name}.files LIMIT 1").collect()[0][0]
    rg = parquet_footer(path).row_group(0)
    cols = [rg.column(i).path_in_schema for i in range(rg.num_columns)]
    total = spark.sql(f"SELECT sum(payload:ms::int) FROM {NS}.{name} WHERE payload:user.plan::string = 'pro'").collect()[0][0]
    print(f"{name:16s} {size:>8,} bytes | {len(cols)} Parquet columns | query result {total}")
    print("   ", [c for c in cols if c.startswith("payload.")][:6], "...")

Same data and the same query result, but the shredded file has typed per-field columns (`payload.typed_value.user.typed_value.plan...`) and is smaller.

### Known limitation in Iceberg 1.12.0: DML on shredded variants
In the lab's stack (Spark 4.1.3 + Iceberg 1.12.0) reads that go through the *vectorized* reader and have to apply deletes or row-lineage columns
failed on shredded variant tables with `Unsupported variant: shredded typed_value primitive`, while plain aggregate reads (above) worked.
The cell below tries a `DELETE`; the workaround is turning vectorization off for that table. If the first attempt prints `OK`, your Iceberg version already has a fix.

In [ ]:
T = f"{NS}.events_shredded"
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('write.delete.mode' = 'merge-on-read')")
try_sql(spark, f"DELETE FROM {T} WHERE id < 10", "DELETE on a shredded variant table (vectorized reads)")
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('read.parquet.vectorization.enabled' = 'false')")
try_sql(spark, f"DELETE FROM {T} WHERE id < 10", "same DELETE with read.parquet.vectorization.enabled=false")
print(spark.sql(f"SELECT count(*) FROM {T}").collect()[0][0], "rows left (49,990 expected)")

## Notes and limits
* Variant needs **format-version 3**. Spark 4.0/4.1 and Trino 483 read it here; other engines may not support v3 yet.
* Use `VARIANT` for genuinely varying shapes. Stable fields are still better as typed columns (cheaper to filter, with column statistics).
* The Iceberg REST server embedded in Hive Metastore 4.2.1 accepts v3 tables and `variant`. It does **not** support the new `geometry`/`geography` types (see `00_whats_new_and_whats_supported`).

In [ ]:
CLEANUP = False   # set True to drop the tables created here
if CLEANUP:
    for t in ["events_v3", "events_plain", "events_shredded"]:
        spark.sql(f"DROP TABLE IF EXISTS {NS}.{t}")